Script 2 — Critic V(s_t) and GAE: Per-Token Credit Assignment
==============================================================
Builds on script 1. The new question:

  The reward R(τ) arrives only at the END.
  But we have T tokens, each with its own log-prob.
  Which token deserves credit?

  REINFORCE answer : all equally  →  high variance
  GAE answer       : use V(s_t) to estimate how much future reward
                     each state was already "worth", then credit only
                     the SURPRISE (δ_t = r_t + γV(s_{t+1}) - V(s_t))

Architecture
------------
  Same tiny LM as script 1 (2 heads, d=16, vocab=8).
  NEW: a Critic head — a small MLP on top of the LM's hidden states
       that outputs a scalar V(s_t) ∈ ℝ at every token position.

Reward
------
  Same rule: +1 for each token == 3 ("c") in the response.
  But now we assign it PER STEP:
    r_t = 1.0  if a_t == "c"
    r_t = 0.0  otherwise
  This makes the credit assignment problem visible:
  a "c" at t=1 should get more credit than one at t=3
  (because it opens more future possibilities).

Steps shown
-----------
  1. Forward pass: LM logits + critic values V(s_t) at every position
  2. Sampling + per-step rewards
  3. TD errors δ_t = r_t + γ V(s_{t+1}) - V(s_t)
  4. GAE advantages  A_t = Σ (γλ)^k δ_{t+k}
  5. Show how A_t differs from flat (R - b) baseline
  6. Actor loss  (policy gradient with GAE advantages)
  7. Critic loss (MSE between V(s_t) and the actual returns G_t)
  8. One joint gradient step on both actor and critic
  9. Verify directions
"""


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

In [2]:
SEP   = "─" * 64
VOCAB = {0:"PAD", 1:"a", 2:"b", 3:"c", 4:"d", 5:"e", 6:"f", 7:"EOS"}
V     = len(VOCAB)
D     = 16
MAX_LEN = 10

def tstr(ids): return " ".join(VOCAB[i] for i in ids)


In [3]:
# ──────────────────────────────────────────────────────────────
# 1.  Model: Actor-Critic sharing a transformer backbone
# ──────────────────────────────────────────────────────────────
class ActorCritic(nn.Module):
    """
    Shared backbone  →  two heads:
      actor  head: linear(d, V)   — logits over next token
      critic head: linear(d, 1)   — scalar value V(s_t)

    Sharing the backbone is common in PPO implementations
    (reduces memory, the representations help both heads).
    """
    def __init__(self):
        super().__init__()
        self.embed   = nn.Embedding(V, D)
        self.pos_enc = nn.Embedding(MAX_LEN, D)
        layer        = nn.TransformerEncoderLayer(
                           d_model=D, nhead=2, dim_feedforward=32,
                           batch_first=True, dropout=0.0)
        self.tf      = nn.TransformerEncoder(layer, num_layers=1)
        # Actor head  (same as script 1)
        self.actor   = nn.Linear(D, V)
        # Critic head  (NEW)
        self.critic  = nn.Sequential(
                           nn.Linear(D, 16),
                           nn.Tanh(),
                           nn.Linear(16, 1))

    def forward(self, token_ids):
        """
        token_ids : (1, T)
        returns
          logits  : (1, T, V)   — actor
          values  : (1, T)      — critic V(s_t) for each position t
        """
        T   = token_ids.shape[1]
        pos = torch.arange(T).unsqueeze(0)
        x   = self.embed(token_ids) + self.pos_enc(pos)
        mask = nn.Transformer.generate_square_subsequent_mask(T)
        h   = self.tf(x, mask=mask, is_causal=True)   # (1, T, D)
        logits = self.actor(h)                          # (1, T, V)
        values = self.critic(h).squeeze(-1)             # (1, T)
        return logits, values

model = ActorCritic()
n_params = sum(p.numel() for p in model.parameters())
print(SEP)
print("MODEL — Actor-Critic")
print(SEP)
print(f"  Total parameters : {n_params}")
print(f"  Backbone         : 1-layer Transformer  d={D}  heads=2")
print(f"  Actor head       : Linear({D}, {V})  → next-token logits")
print(f"  Critic head      : Linear({D},16) → Tanh → Linear(16,1) → V(s_t)")
print()

────────────────────────────────────────────────────────────────
MODEL — Actor-Critic
────────────────────────────────────────────────────────────────
  Total parameters : 2937
  Backbone         : 1-layer Transformer  d=16  heads=2
  Actor head       : Linear(16, 8)  → next-token logits
  Critic head      : Linear(16,16) → Tanh → Linear(16,1) → V(s_t)



In [4]:
# ──────────────────────────────────────────────────────────────
# 2.  Autoregressive rollout — collect tokens, log-probs, values, rewards
# ──────────────────────────────────────────────────────────────
PROMPT   = [1, 2]    # "a b"
MAX_GEN  = 4
EOS_ID   = 7
TARGET   = 3         # "c"
GAMMA    = 0.99      # discount
LAM      = 0.95      # GAE lambda

print(SEP)
print("STEP 1 — ROLLOUT  (tokens + values + rewards)")
print(SEP)
print(f"  Prompt : {tstr(PROMPT)}")
print()

model.eval()
ids = torch.tensor([PROMPT])   # (1, T_prompt)

sampled_tokens = []
log_probs      = []   # log π(a_t | s_t)
values         = []   # V(s_t)  at the decision step
rewards        = []   # r_t  (per-step, known immediately here)

for step in range(MAX_GEN):
    logits, vals = model(ids)         # (1,T,V), (1,T)
    # Last position = the one predicting the NEXT token
    next_logits  = logits[0, -1, :]   # (V,)
    next_val     = vals[0, -1]        # scalar — V(s_t) before sampling

    probs      = F.softmax(next_logits, dim=-1)
    action     = torch.multinomial(probs, 1).item()
    log_prob   = F.log_softmax(next_logits, dim=-1)[action]
    reward     = 1.0 if action == TARGET else 0.0

    sampled_tokens.append(action)
    log_probs.append(log_prob)
    values.append(next_val)
    rewards.append(reward)

    print(f"  t={step}  state='{tstr(ids[0].tolist())}'")
    print(f"       V(s_{step})  = {next_val.item():+.4f}   (critic estimate)")
    print(f"       sampled  → '{VOCAB[action]}'  "
          f"log_prob={log_prob.item():.4f}  r_t={reward:.1f}")
    print()

    ids = torch.cat([ids, torch.tensor([[action]])], dim=1)

    if action == EOS_ID:
        print("  [EOS — episode ends]")
        print()
        break

T = len(sampled_tokens)   # actual response length

# Bootstrap value at the terminal state (= 0 if EOS, else V(s_T))
_, vals_final = model(ids)
bootstrap = 0.0 if sampled_tokens[-1] == EOS_ID else vals_final[0, -1].item()

print(f"  Response : '{tstr(sampled_tokens)}'")
print(f"  Total R  : {sum(rewards):.1f}  (number of 'c' tokens)")
print()

────────────────────────────────────────────────────────────────
STEP 1 — ROLLOUT  (tokens + values + rewards)
────────────────────────────────────────────────────────────────
  Prompt : a b

  t=0  state='a b'
       V(s_0)  = -0.0499   (critic estimate)
       sampled  → 'e'  log_prob=-2.3530  r_t=0.0

  t=1  state='a b e'
       V(s_1)  = -0.0251   (critic estimate)
       sampled  → 'b'  log_prob=-1.5840  r_t=0.0

  t=2  state='a b e b'
       V(s_2)  = +0.0409   (critic estimate)
       sampled  → 'e'  log_prob=-2.8756  r_t=0.0

  t=3  state='a b e b e'
       V(s_3)  = +0.0812   (critic estimate)
       sampled  → 'EOS'  log_prob=-1.6954  r_t=0.0

  [EOS — episode ends]

  Response : 'e b e EOS'
  Total R  : 0.0  (number of 'c' tokens)



In [5]:
# ──────────────────────────────────────────────────────────────
# 3.  TD errors  δ_t
# ──────────────────────────────────────────────────────────────
print(SEP)
print("STEP 2 — TD ERRORS  δ_t = r_t + γ·V(s_{t+1}) - V(s_t)")
print(SEP)
print("""
  Intuition:
    V(s_t)        = what the critic EXPECTED to get from state t onward
    r_t + γV(s_{t+1}) = what actually happened (one real step + new estimate)
    δ_t           = the SURPRISE — better or worse than expected?

    δ_t > 0  →  things went better than the critic predicted at t
    δ_t < 0  →  things went worse
""")

values_np = [v.item() for v in values]
# append bootstrap for V(s_T)
values_ext = values_np + [bootstrap]

deltas = []
for t in range(T):
    d = rewards[t] + GAMMA * values_ext[t+1] - values_ext[t]
    deltas.append(d)
    print(f"  δ_{t} = r_{t} + γ·V(s_{t+1}) - V(s_{t})")
    print(f"      = {rewards[t]:.1f} + {GAMMA}·{values_ext[t+1]:+.4f}"
          f" - {values_ext[t]:+.4f} = {d:+.4f}")
print()


────────────────────────────────────────────────────────────────
STEP 2 — TD ERRORS  δ_t = r_t + γ·V(s_{t+1}) - V(s_t)
────────────────────────────────────────────────────────────────

  Intuition:
    V(s_t)        = what the critic EXPECTED to get from state t onward
    r_t + γV(s_{t+1}) = what actually happened (one real step + new estimate)
    δ_t           = the SURPRISE — better or worse than expected?

    δ_t > 0  →  things went better than the critic predicted at t
    δ_t < 0  →  things went worse

  δ_0 = r_0 + γ·V(s_1) - V(s_0)
      = 0.0 + 0.99·-0.0251 - -0.0499 = +0.0250
  δ_1 = r_1 + γ·V(s_2) - V(s_1)
      = 0.0 + 0.99·+0.0409 - -0.0251 = +0.0656
  δ_2 = r_2 + γ·V(s_3) - V(s_2)
      = 0.0 + 0.99·+0.0812 - +0.0409 = +0.0395
  δ_3 = r_3 + γ·V(s_4) - V(s_3)
      = 0.0 + 0.99·+0.0000 - +0.0812 = -0.0812



In [6]:
# ──────────────────────────────────────────────────────────────
# 4.  GAE advantages  A_t
# ──────────────────────────────────────────────────────────────
print(SEP)
print("STEP 3 — GAE ADVANTAGES  A_t = Σ_{k≥0} (γλ)^k · δ_{t+k}")
print(SEP)
print(f"""
  γ = {GAMMA}  (discount — future rewards worth slightly less)
  λ = {LAM}  (GAE smoothing — interpolates MC ↔ TD)

  λ=0 → A_t = δ_t            (pure TD, low variance, more bias)
  λ=1 → A_t = G_t - V(s_t)   (Monte Carlo, unbiased, high variance)
  λ={LAM} → weighted sum of future δ's

  Computed BACKWARDS from the last step:
""")

advantages = [0.0] * T
gae = 0.0
for t in reversed(range(T)):
    gae = deltas[t] + GAMMA * LAM * gae
    advantages[t] = gae

for t in range(T):
    print(f"  A_{t}  token='{VOCAB[sampled_tokens[t]]}'  "
          f"δ_{t}={deltas[t]:+.4f}  A_{t}={advantages[t]:+.4f}")
print()

# Compare with flat baseline from script 1
R_total = sum(rewards)
flat_b  = 0.667   # same simulated batch mean as script 1
flat_A  = R_total - flat_b
print(f"  For comparison — flat baseline (script 1):")
print(f"    A = R - b = {R_total:.1f} - {flat_b:.3f} = {flat_A:+.4f}  "
      f"(same for ALL tokens)")
print()
print(f"  GAE advantages per token:")
for t in range(T):
    print(f"    t={t}  '{VOCAB[sampled_tokens[t]]}'  A_t={advantages[t]:+.4f}")
print()
print("  GAE differentiates between tokens — early tokens that")
print("  led to good outcomes get different credit than late ones.")
print()


────────────────────────────────────────────────────────────────
STEP 3 — GAE ADVANTAGES  A_t = Σ_{k≥0} (γλ)^k · δ_{t+k}
────────────────────────────────────────────────────────────────

  γ = 0.99  (discount — future rewards worth slightly less)
  λ = 0.95  (GAE smoothing — interpolates MC ↔ TD)

  λ=0 → A_t = δ_t            (pure TD, low variance, more bias)
  λ=1 → A_t = G_t - V(s_t)   (Monte Carlo, unbiased, high variance)
  λ=0.95 → weighted sum of future δ's

  Computed BACKWARDS from the last step:

  A_0  token='e'  δ_0=+0.0250  A_0=+0.0541
  A_1  token='b'  δ_1=+0.0656  A_1=+0.0309
  A_2  token='e'  δ_2=+0.0395  A_2=-0.0369
  A_3  token='EOS'  δ_3=-0.0812  A_3=-0.0812

  For comparison — flat baseline (script 1):
    A = R - b = 0.0 - 0.667 = -0.6670  (same for ALL tokens)

  GAE advantages per token:
    t=0  'e'  A_t=+0.0541
    t=1  'b'  A_t=+0.0309
    t=2  'e'  A_t=-0.0369
    t=3  'EOS'  A_t=-0.0812

  GAE differentiates between tokens — early tokens that
  led to good o

In [7]:
# ──────────────────────────────────────────────────────────────
# 5.  Returns G_t  (target for critic)
# ──────────────────────────────────────────────────────────────
print(SEP)
print("STEP 4 — RETURNS G_t  (what the critic should have predicted)")
print(SEP)
print("""
  G_t = r_t + γ·r_{t+1} + γ²·r_{t+2} + ...   (discounted sum from t)

  The critic loss will be:  L_critic = Σ_t (V(s_t) - G_t)²
  This trains V to predict future returns accurately,
  so that future δ_t = r_t + γV(s_{t+1}) - V(s_t) shrinks → less variance.
""")

returns = [0.0] * T
G = bootstrap
for t in reversed(range(T)):
    G = rewards[t] + GAMMA * G
    returns[t] = G

for t in range(T):
    print(f"  G_{t} = {returns[t]:.4f}   V(s_{t}) = {values_np[t]:+.4f}   "
          f"error = {returns[t] - values_np[t]:+.4f}")
print()

────────────────────────────────────────────────────────────────
STEP 4 — RETURNS G_t  (what the critic should have predicted)
────────────────────────────────────────────────────────────────

  G_t = r_t + γ·r_{t+1} + γ²·r_{t+2} + ...   (discounted sum from t)

  The critic loss will be:  L_critic = Σ_t (V(s_t) - G_t)²
  This trains V to predict future returns accurately,
  so that future δ_t = r_t + γV(s_{t+1}) - V(s_t) shrinks → less variance.

  G_0 = 0.0000   V(s_0) = -0.0499   error = +0.0499
  G_1 = 0.0000   V(s_1) = -0.0251   error = +0.0251
  G_2 = 0.0000   V(s_2) = +0.0409   error = -0.0409
  G_3 = 0.0000   V(s_3) = +0.0812   error = -0.0812



In [8]:
# ──────────────────────────────────────────────────────────────
# 6.  Actor loss + Critic loss
# ──────────────────────────────────────────────────────────────
print(SEP)
print("STEP 5 — LOSSES")
print(SEP)
print("""
  Actor  loss = -Σ_t  A_t · log π_θ(a_t | s_t)
    Maximize advantage-weighted log-probs.
    A_t > 0 → push this token UP
    A_t < 0 → push this token DOWN

  Critic loss =  Σ_t  (V(s_t) - G_t)²
    Standard MSE regression target.
    Better V → better δ → better A → lower variance.

  Joint loss = actor_loss + c_v · critic_loss
    c_v is a coefficient (typically 0.5).
""")

C_V = 0.5   # critic coefficient

# Recompute with gradients
model.train()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
optimizer.zero_grad()

ids_g = torch.tensor([PROMPT])
log_probs_g = []
values_g    = []

for t, action in enumerate(sampled_tokens):
    logits, vals = model(ids_g)
    lp  = F.log_softmax(logits[0, -1, :], dim=-1)[action]
    val = vals[0, -1]
    log_probs_g.append(lp)
    values_g.append(val)
    ids_g = torch.cat([ids_g, torch.tensor([[action]])], dim=1)

adv_t  = torch.tensor(advantages, dtype=torch.float32)
ret_t  = torch.tensor(returns,    dtype=torch.float32)
lp_t   = torch.stack(log_probs_g)
val_t  = torch.stack(values_g)

actor_loss  = -(adv_t * lp_t).sum()
critic_loss =  F.mse_loss(val_t, ret_t)
joint_loss  =  actor_loss + C_V * critic_loss

print(f"  Per-token breakdown:")
print(f"  {'t':>3}  {'token':>5}  {'A_t':>8}  {'log_p':>8}  "
      f"{'A·logp':>9}  {'V(s_t)':>8}  {'G_t':>8}  {'(V-G)²':>8}")
print(f"  {'─'*3}  {'─'*5}  {'─'*8}  {'─'*8}  {'─'*9}  {'─'*8}  {'─'*8}  {'─'*8}")
for t in range(T):
    a   = adv_t[t].item()
    lp  = lp_t[t].item()
    v   = val_t[t].item()
    g   = ret_t[t].item()
    print(f"  {t:>3}  {VOCAB[sampled_tokens[t]]:>5}  {a:>+8.4f}  {lp:>8.4f}  "
          f"{a*lp:>+9.4f}  {v:>+8.4f}  {g:>8.4f}  {(v-g)**2:>8.4f}")
print()
print(f"  actor_loss  = -Σ(A_t · log_p)  = {actor_loss.item():+.4f}")
print(f"  critic_loss = MSE(V, G)         = {critic_loss.item():+.4f}")
print(f"  joint_loss  = actor + 0.5·critic= {joint_loss.item():+.4f}")
print()

────────────────────────────────────────────────────────────────
STEP 5 — LOSSES
────────────────────────────────────────────────────────────────

  Actor  loss = -Σ_t  A_t · log π_θ(a_t | s_t)
    Maximize advantage-weighted log-probs.
    A_t > 0 → push this token UP
    A_t < 0 → push this token DOWN

  Critic loss =  Σ_t  (V(s_t) - G_t)²
    Standard MSE regression target.
    Better V → better δ → better A → lower variance.

  Joint loss = actor_loss + c_v · critic_loss
    c_v is a coefficient (typically 0.5).

  Per-token breakdown:
    t  token       A_t     log_p     A·logp    V(s_t)       G_t    (V-G)²
  ───  ─────  ────────  ────────  ─────────  ────────  ────────  ────────
    0      e   +0.0541   -2.3530    -0.1273   -0.0499    0.0000    0.0025
    1      b   +0.0309   -1.5840    -0.0490   -0.0251    0.0000    0.0006
    2      e   -0.0369   -2.8756    +0.1061   +0.0409    0.0000    0.0017
    3    EOS   -0.0812   -1.6954    +0.1377   +0.0812    0.0000    0.0066

  actor_l

In [9]:
# ──────────────────────────────────────────────────────────────
# 7.  Backward + step
# ──────────────────────────────────────────────────────────────
print(SEP)
print("STEP 6 — BACKWARD + OPTIMIZER STEP")
print(SEP)

joint_loss.backward()

print("  Gradient norms (actor head vs critic head vs backbone):")
actor_gnorm  = model.actor.weight.grad.norm().item()
critic_gnorm = sum(p.grad.norm().item()**2
                   for p in model.critic.parameters()
                   if p.grad is not None) ** 0.5
backbone_gnorm = sum(p.grad.norm().item()**2
                     for name, p in model.named_parameters()
                     if p.grad is not None
                     and 'actor' not in name
                     and 'critic' not in name) ** 0.5
print(f"    actor  head : {actor_gnorm:.6f}")
print(f"    critic head : {critic_gnorm:.6f}")
print(f"    backbone    : {backbone_gnorm:.6f}")
print()

optimizer.step()
print("  optimizer.step() done.")
print()

────────────────────────────────────────────────────────────────
STEP 6 — BACKWARD + OPTIMIZER STEP
────────────────────────────────────────────────────────────────
  Gradient norms (actor head vs critic head vs backbone):
    actor  head : 0.343489
    critic head : 0.064530
    backbone    : 0.256531

  optimizer.step() done.



In [10]:
# ──────────────────────────────────────────────────────────────
# 8.  Verification
# ──────────────────────────────────────────────────────────────
print(SEP)
print("STEP 7 — VERIFICATION")
print(SEP)
print("  Check that log-probs and values moved in the right direction.\n")

model.eval()
ids_v = torch.tensor([PROMPT])

print(f"  {'t':>3}  {'tok':>5}  {'A_t':>8}  "
      f"{'lp before':>10}  {'lp after':>10}  {'Δlp':>8}  "
      f"{'V before':>9}  {'V after':>9}  {'ΔV':>8}")
print(f"  {'─'*3}  {'─'*5}  {'─'*8}  {'─'*10}  {'─'*10}  {'─'*8}  {'─'*9}  {'─'*9}  {'─'*8}")

with torch.no_grad():
    for t, action in enumerate(sampled_tokens):
        logits_new, vals_new = model(ids_v)
        lp_new  = F.log_softmax(logits_new[0, -1, :], dim=-1)[action].item()
        val_new = vals_new[0, -1].item()

        lp_old  = lp_t[t].item()
        val_old = val_t[t].item()
        a       = adv_t[t].item()
        G_t     = ret_t[t].item()

        dlp  = lp_new - lp_old
        dval = val_new - val_old
        exp_lp  = "↑" if a > 0 else ("↓" if a < 0 else "=")
        exp_val = "↑" if G_t > val_old else ("↓" if G_t < val_old else "=")

        print(f"  {t:>3}  {VOCAB[action]:>5}  {a:>+8.4f}  "
              f"{lp_old:>10.4f}  {lp_new:>10.4f}  {dlp:>+8.4f}{exp_lp}  "
              f"{val_old:>+9.4f}  {val_new:>+9.4f}  {dval:>+8.4f}{exp_val}")

        ids_v = torch.cat([ids_v, torch.tensor([[action]])], dim=1)

print()
print("  ↑ = moved in expected direction    ↓ = moved in expected direction (down)")
print()


────────────────────────────────────────────────────────────────
STEP 7 — VERIFICATION
────────────────────────────────────────────────────────────────
  Check that log-probs and values moved in the right direction.

    t    tok       A_t   lp before    lp after       Δlp   V before    V after        ΔV
  ───  ─────  ────────  ──────────  ──────────  ────────  ─────────  ─────────  ────────
    0      e   +0.0541     -2.3530     -2.3371   +0.0158↑    -0.0499    -0.0411   +0.0088↑
    1      b   +0.0309     -1.5840     -1.5440   +0.0400↑    -0.0251    -0.0256   -0.0004↑
    2      e   -0.0369     -2.8756     -2.8959   -0.0203↓    +0.0409    +0.0320   -0.0089↓
    3    EOS   -0.0812     -1.6954     -1.7836   -0.0882↓    +0.0812    +0.0475   -0.0337↓

  ↑ = moved in expected direction    ↓ = moved in expected direction (down)



In [11]:
# ──────────────────────────────────────────────────────────────
# 9.  Summary
# ──────────────────────────────────────────────────────────────
print(SEP)
print("CONCEPTUAL SUMMARY")
print(SEP)
print(f"""
  WHY A CRITIC?
  ─────────────
  REINFORCE uses the flat baseline  b = mean(R_batch).
  It's the same number for every token in the sequence.

  The critic V(s_t) gives a DIFFERENT baseline per token:
    "from this specific state (context so far), how much reward do I expect?"

  This means:
    - If the context already looks bad, a mediocre token still gets A_t ≈ 0
    - If the context looks great, the same token gets A_t < 0 (below expectation)
    - Credit assignment is sharper and more accurate

  GAE (γ={GAMMA}, λ={LAM})
  ─────────────────
    A_t = sum_k (gamma*lambda)^k * delta_(t+k)
    delta_t = r_t + gamma*V(s_t+1) - V(s_t)   <- TD error = "surprise"

    Early tokens accumulate more future δ's (longer lookahead).
    Late  tokens have fewer terms (less lookahead, lower variance).

  TWO LOSSES, ONE BACKWARD PASS
  ──────────────────────────────
    actor_loss  = -Σ_t  A_t · log π(a_t|s_t)   → train the LM
    critic_loss =  Σ_t  (V(s_t) - G_t)²        → train the value head
    joint_loss  = actor_loss + 0.5 · critic_loss

  As the critic improves across training steps:
    V(s_t) → true E[G_t]
    δ_t    → smaller (less surprise)
    A_t    → lower variance
    actor gradient → more reliable signal

  Next step → PPO clipping:
    Prevents the actor from taking steps so large that
    the new policy is totally different from the one that collected the data.
""")


────────────────────────────────────────────────────────────────
CONCEPTUAL SUMMARY
────────────────────────────────────────────────────────────────

  WHY A CRITIC?
  ─────────────
  REINFORCE uses the flat baseline  b = mean(R_batch).
  It's the same number for every token in the sequence.

  The critic V(s_t) gives a DIFFERENT baseline per token:
    "from this specific state (context so far), how much reward do I expect?"

  This means:
    - If the context already looks bad, a mediocre token still gets A_t ≈ 0
    - If the context looks great, the same token gets A_t < 0 (below expectation)
    - Credit assignment is sharper and more accurate

  GAE (γ=0.99, λ=0.95)
  ─────────────────
    A_t = sum_k (gamma*lambda)^k * delta_(t+k)
    delta_t = r_t + gamma*V(s_t+1) - V(s_t)   <- TD error = "surprise"

    Early tokens accumulate more future δ's (longer lookahead).
    Late  tokens have fewer terms (less lookahead, lower variance).

  TWO LOSSES, ONE BACKWARD PASS
  ──────────────